# Model 1: PPG → Blood Pressure Estimation
**Dataset**: VitalDB (PhysioNet)

This notebook trains a deep learning model to estimate Systolic (SBP) and Diastolic (DBP) blood pressure from PPG waveform segments.

**Architecture**: 1D CNN + BiLSTM + Attention

**Input**: 625-sample PPG window (5 seconds @ 125 Hz)

**Output**: [SBP, DBP] in mmHg

In [ ]:
# Install dependencies
! pip install vitaldb wfdb torch torchvision torchaudio scikit-learn matplotlib numpy pandas tqdm

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import vitaldb
from tqdm import tqdm
import warnings
import os
import pickle

os.makedirs(os.path.join('..', 'Training plots'), exist_ok=True)

warnings.filterwarnings('ignore')

# ─── Device ───────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ─── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## 1. Data Loading from VitalDB (Local Files)

In [ ]:
# ─── Configuration ─────────────────────────────────────────────────────────────
# !! Set this to the root folder of your extracted VitalDB zip (6 388 .vital files) !!
VITALDB_DIR  = r'data/vitaldb'   # e.g. r'C:\datasets\vitaldb' or '/data/vitaldb'
CACHE_DIR    = 'ppg_bp_cache'    # folder where per-chunk .pkl files are stored

FS           = 125      # Target sampling frequency (Hz)
WIN_SEC      = 5        # Window size in seconds
WIN_SAMPLES  = FS * WIN_SEC  # 625 samples per window
STRIDE_SEC   = 2        # Stride between consecutive windows
STRIDE       = FS * STRIDE_SEC

PPG_TRACK    = 'SNUADC/PLETH'       # PPG waveform track name inside .vital files
SBP_TRACK    = 'Solar8000/ART_SBP'  # Systolic BP numeric
DBP_TRACK    = 'Solar8000/ART_DBP'  # Diastolic BP numeric

# Valid BP range (filter artifacts)
SBP_MIN, SBP_MAX = 70, 200
DBP_MIN, DBP_MAX = 40, 130

# ── Large-dataset settings ────────────────────────────────────────────────────
FILES_PER_CHUNK = 200    # Number of .vital files processed per cache chunk
                         # Each chunk ≈ 200 files × ~2 000 windows × 625 floats ≈ ~1 GB
                         # Reduce to 100 if you have less than 16 GB RAM
NUM_WORKERS     = 4      # DataLoader worker processes (set to 0 on Windows if errors occur)
BATCH_SIZE      = 512    # Larger batch = faster epoch on big data; reduce if OOM


In [ ]:
import glob
from vitaldb import VitalFile

# ─────────────────────────────────────────────────────────────────────────────
# Step 1 – Build chunked cache (skipped if already done)
# Each chunk is saved as CACHE_DIR/chunk_XXXX.pkl
# This avoids loading all 6 388 files × millions of windows into RAM at once.
# ─────────────────────────────────────────────────────────────────────────────
os.makedirs(CACHE_DIR, exist_ok=True)

vital_files = sorted(glob.glob(os.path.join(VITALDB_DIR, '**', '*.vital'), recursive=True))
if not vital_files:
    raise FileNotFoundError(
        f'No .vital files found under "{VITALDB_DIR}".\n'
        f'Please extract the VitalDB zip and set VITALDB_DIR correctly.'
    )
print(f'Found {len(vital_files)} .vital files')

# Split file list into chunks
file_chunks = [vital_files[i:i+FILES_PER_CHUNK]
               for i in range(0, len(vital_files), FILES_PER_CHUNK)]
print(f'Will create {len(file_chunks)} cache chunks of up to {FILES_PER_CHUNK} files each')


def process_chunk(fpath_list):
    """Extract PPG windows + BP labels from a list of .vital files."""
    chunk_ppg, chunk_sbp, chunk_dbp = [], [], []
    for fpath in fpath_list:
        try:
            vf      = VitalFile(fpath, [PPG_TRACK, SBP_TRACK, DBP_TRACK])
            ppg_arr = vf.to_numpy([PPG_TRACK], interval=1/FS)
            sbp_arr = vf.to_numpy([SBP_TRACK], interval=1)
            dbp_arr = vf.to_numpy([DBP_TRACK], interval=1)
            if ppg_arr is None or sbp_arr is None or dbp_arr is None:
                continue
            ppg_arr = ppg_arr[:, 0]
            sbp_arr = sbp_arr[:, 0]
            dbp_arr = dbp_arr[:, 0]
            n_windows = (len(ppg_arr) - WIN_SAMPLES) // STRIDE
            for w in range(n_windows):
                start    = w * STRIDE
                ppg_win  = ppg_arr[start:start + WIN_SAMPLES]
                t_center = (start + WIN_SAMPLES // 2) // FS
                if t_center >= len(sbp_arr) or t_center >= len(dbp_arr):
                    continue
                sbp_val = sbp_arr[t_center]
                dbp_val = dbp_arr[t_center]
                if np.any(np.isnan(ppg_win)) or np.isnan(sbp_val) or np.isnan(dbp_val):
                    continue
                if not (SBP_MIN <= sbp_val <= SBP_MAX and DBP_MIN <= dbp_val <= DBP_MAX):
                    continue
                if np.std(ppg_win) < 1e-4:
                    continue
                chunk_ppg.append(ppg_win.astype(np.float32))
                chunk_sbp.append(float(sbp_val))
                chunk_dbp.append(float(dbp_val))
        except Exception:
            continue
    return (np.array(chunk_ppg, dtype=np.float32),
            np.array(chunk_sbp, dtype=np.float32),
            np.array(chunk_dbp, dtype=np.float32))


# Build any missing chunks
total_windows = 0
chunk_files   = []
for ci, flist in enumerate(tqdm(file_chunks, desc='Building cache chunks')):
    cpath = os.path.join(CACHE_DIR, f'chunk_{ci:04d}.pkl')
    chunk_files.append(cpath)
    if os.path.exists(cpath):
        with open(cpath, 'rb') as f:
            c = pickle.load(f)
        total_windows += len(c[0])
        continue                          # already cached – skip processing
    ppg_c, sbp_c, dbp_c = process_chunk(flist)
    with open(cpath, 'wb') as f:
        pickle.dump((ppg_c, sbp_c, dbp_c), f)
    total_windows += len(ppg_c)

print(f'\nCache ready  – {len(chunk_files)} chunk files')
print(f'Total windows across all chunks: {total_windows:,}')


# ─────────────────────────────────────────────────────────────────────────────
# Step 2 – Fit BP scaler on a representative sample (first chunk only)
# ─────────────────────────────────────────────────────────────────────────────
import joblib
from sklearn.preprocessing import StandardScaler

SCALER_FILE = 'bp_scaler.pkl'
if os.path.exists(SCALER_FILE):
    bp_scaler = joblib.load(SCALER_FILE)
    print('BP scaler loaded from cache.')
else:
    print('Fitting BP scaler on first chunk...')
    with open(chunk_files[0], 'rb') as f:
        sample_ppg, sample_sbp, sample_dbp = pickle.load(f)
    bp_labels_sample = np.stack([sample_sbp, sample_dbp], axis=1)
    bp_scaler = StandardScaler().fit(bp_labels_sample)
    joblib.dump(bp_scaler, SCALER_FILE)
    print(f'BP scaler fitted and saved to {SCALER_FILE}')


# ─────────────────────────────────────────────────────────────────────────────
# Step 3 – ChunkedPPGDataset: streams chunks on-the-fly, never loads all at once
# ─────────────────────────────────────────────────────────────────────────────
class ChunkedPPGDataset(Dataset):
    """
    Memory-efficient Dataset for 6 388 VitalDB cases.
    Loads one chunk at a time — only the active chunk lives in RAM.
    Compatible with PyTorch DataLoader (num_workers > 0 supported).
    """
    def __init__(self, chunk_files, bp_scaler):
        self.chunk_files = chunk_files
        self.bp_scaler   = bp_scaler
        # Build an index: (chunk_idx, local_idx) for every window
        self.index = []          # list of (chunk_idx, local_idx)
        self.chunk_sizes = []
        for ci, cpath in enumerate(chunk_files):
            with open(cpath, 'rb') as f:
                d = pickle.load(f)
            n = len(d[0])
            self.chunk_sizes.append(n)
            self.index.extend((ci, li) for li in range(n))
        # Cache for the currently loaded chunk
        self._loaded_ci   = None
        self._loaded_data = None

    def __len__(self):
        return len(self.index)

    def _load_chunk(self, ci):
        if self._loaded_ci != ci:
            with open(self.chunk_files[ci], 'rb') as f:
                ppg_c, sbp_c, dbp_c = pickle.load(f)
            # Normalise PPG per window
            mu  = ppg_c.mean(axis=1, keepdims=True)
            std = ppg_c.std(axis=1,  keepdims=True) + 1e-8
            ppg_n = (ppg_c - mu) / std
            # Normalise BP
            bp_raw = np.stack([sbp_c, dbp_c], axis=1)
            bp_n   = self.bp_scaler.transform(bp_raw).astype(np.float32)
            self._loaded_ci   = ci
            self._loaded_data = (ppg_n, bp_n)
        return self._loaded_data

    def __getitem__(self, idx):
        ci, li = self.index[idx]
        ppg_n, bp_n = self._load_chunk(ci)
        x = torch.tensor(ppg_n[li][np.newaxis, :], dtype=torch.float32)  # (1, 625)
        y = torch.tensor(bp_n[li],                 dtype=torch.float32)  # (2,)
        return x, y


full_dataset = ChunkedPPGDataset(chunk_files, bp_scaler)
n_total  = len(full_dataset)
n_train  = int(0.70 * n_total)
n_val    = int(0.15 * n_total)
n_test   = n_total - n_train - n_val

train_set, val_set, test_set = random_split(
    full_dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=(NUM_WORKERS > 0))
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=(NUM_WORKERS > 0))
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=(NUM_WORKERS > 0))

print(f'\nDataset split  →  Train: {n_train:,} | Val: {n_val:,} | Test: {n_test:,}')
print(f'Batches/epoch  →  Train: {len(train_loader):,} | Val: {len(val_loader):,}')

# Keep these for the inference helper at the bottom
WIN_SAMP = WIN_SAMPLES
FS_PPG   = FS


## 2. Preprocessing & Dataset

In [ ]:
# Normalization is handled inside ChunkedPPGDataset.__getitem__.
# The BP scaler was fitted and saved in the data loading cell above.
# Re-load it here so downstream cells (evaluation, inference) can use it.
import joblib
from sklearn.preprocessing import StandardScaler

bp_scaler = joblib.load('bp_scaler.pkl')
print(f'BP scaler loaded. Mean SBP={bp_scaler.mean_[0]:.1f}, Mean DBP={bp_scaler.mean_[1]:.1f}')

# Quick sanity: load one sample to verify shapes
sample_x, sample_y = full_dataset[0]
print(f'Sample PPG tensor : {sample_x.shape}  (expected [1, 625])')
print(f'Sample BP tensor  : {sample_y.shape}   (expected [2])')
print(f'\nDataLoader summary:')
print(f'  Batch size : {BATCH_SIZE}')
print(f'  Num workers: {NUM_WORKERS}')
print(f'  Train batches/epoch: {len(train_loader):,}')


In [ ]:
# ─── Plot a sample PPG window and the BP distribution (from first cache chunk)
with open(chunk_files[0], 'rb') as f:
    _ppg0, _sbp0, _dbp0 = pickle.load(f)

_mu  = _ppg0[0].mean()
_std = _ppg0[0].std() + 1e-8
_ppg0_norm = (_ppg0[0] - _mu) / _std

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
t = np.arange(WIN_SAMPLES) / FS
axes[0].plot(t, _ppg0_norm)
axes[0].set_title('Sample Normalized PPG Window (Case 1)')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude (normalized)')
axes[0].grid(True, alpha=0.3)

axes[1].hist(_sbp0, bins=50, alpha=0.7, label='SBP', color='red')
axes[1].hist(_dbp0, bins=50, alpha=0.7, label='DBP', color='blue')
axes[1].set_title(f'BP Distribution (chunk 0, {len(_sbp0):,} windows)')
axes[1].set_xlabel('mmHg')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
del _ppg0, _sbp0, _dbp0


In [ ]:
# Dataset and DataLoaders were created in the data-loading cell above.
# Run this cell only if you want to re-print the split sizes.
print(f'Total windows : {n_total:,}')
print(f'Train         : {n_train:,}')
print(f'Val           : {n_val:,}')
print(f'Test          : {n_test:,}')


## 3. Model Architecture: CNN-BiLSTM with Attention

In [ ]:
class ChannelAttention(nn.Module):
    """Squeeze-and-Excitation style temporal attention."""
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )

    def forward(self, x):  # x: (B, C, T)
        w = x.mean(dim=2)       # global average pooling → (B, C)
        w = self.fc(w)          # → (B, C)
        return x * w.unsqueeze(2)


class ResBlock1D(nn.Module):
    """1D Residual block with attention."""
    def __init__(self, channels, kernel_size=7):
        super().__init__()
        pad = kernel_size // 2
        self.conv = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size, padding=pad),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Conv1d(channels, channels, kernel_size, padding=pad),
            nn.BatchNorm1d(channels),
        )
        self.attn = ChannelAttention(channels)
        self.act  = nn.GELU()

    def forward(self, x):
        return self.act(self.attn(self.conv(x)) + x)


class PPGtoBP(nn.Module):
    """
    1D CNN feature extractor → BiLSTM temporal modelling → MLP regressor.
    Input : (B, 1, 625)
    Output: (B, 2)  [SBP_norm, DBP_norm]
    """
    def __init__(self, win_len=625, lstm_hidden=128, lstm_layers=2, dropout=0.3):
        super().__init__()

        # ── Stem ──────────────────────────────────────────────────────────────
        self.stem = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=15, padding=7, stride=2),  # → 312
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        # ── Residual CNN blocks ───────────────────────────────────────────────
        self.layer1 = nn.Sequential(
            nn.Conv1d(32, 64, 3, stride=2, padding=1),  # → 156
            ResBlock1D(64),
            ResBlock1D(64),
        )
        self.layer2 = nn.Sequential(
            nn.Conv1d(64, 128, 3, stride=2, padding=1),  # → 78
            ResBlock1D(128),
            ResBlock1D(128),
        )
        self.layer3 = nn.Sequential(
            nn.Conv1d(128, 256, 3, stride=2, padding=1),  # → 39
            ResBlock1D(256),
        )

        # ── BiLSTM ────────────────────────────────────────────────────────────
        self.bilstm = nn.LSTM(
            input_size=256, hidden_size=lstm_hidden,
            num_layers=lstm_layers, batch_first=True,
            bidirectional=True, dropout=dropout
        )

        # ── Self-attention over LSTM outputs ──────────────────────────────────
        self.query = nn.Linear(lstm_hidden * 2, 1)

        # ── Regressor ─────────────────────────────────────────────────────────
        self.regressor = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):           # x: (B,1,625)
        x = self.stem(x)            # (B,32,312)
        x = self.layer1(x)          # (B,64,156)
        x = self.layer2(x)          # (B,128,78)
        x = self.layer3(x)          # (B,256,39)
        x = x.permute(0, 2, 1)      # (B,39,256) – time-first for LSTM
        x, _ = self.bilstm(x)       # (B,39,256)
        # Attention-weighted pooling
        w = torch.softmax(self.query(x), dim=1)  # (B,39,1)
        x = (x * w).sum(dim=1)      # (B,256)
        return self.regressor(x)    # (B,2)


model = PPGtoBP().to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {total_params:,}')
print(model)

## 4. Training

In [ ]:
import time

EPOCHS    = 50
LR        = 1e-3
PATIENCE  = 10

criterion  = nn.SmoothL1Loss()   # Huber loss – robust to outliers
optimizer  = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


def run_epoch(loader, train=True):
    model.train(train)
    total_loss = 0
    with torch.set_grad_enabled(train):
        for ppg, bp in loader:
            ppg, bp = ppg.to(device), bp.to(device)
            pred = model(ppg)
            loss = criterion(pred, bp)
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item() * len(ppg)
    return total_loss / len(loader.dataset)


train_losses, val_losses = [], []
best_val = float('inf')
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    t0       = time.time()
    tr_loss  = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader,   train=False)
    elapsed  = time.time() - t0
    scheduler.step()

    train_losses.append(tr_loss)
    val_losses.append(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), 'best_ppg_to_bp.pth')
        patience_counter = 0
        flag = ' ✓ saved'
    else:
        patience_counter += 1
        flag = ''

    if epoch % 5 == 0 or patience_counter == 0:
        print(f'Epoch {epoch:3d} | Train: {tr_loss:.4f} | Val: {val_loss:.4f} '
              f'| LR: {scheduler.get_last_lr()[0]:.2e} | {elapsed:.0f}s{flag}')

    if patience_counter >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'\nBest val loss: {best_val:.4f}')
print(f'Model saved  : best_ppg_to_bp.pth')


## 5. Evaluation

In [ ]:
# ─── Loss curves ──────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses,   label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Huber Loss')
plt.title('Training Curves – PPG → BP')
plt.legend()
plt.grid(True)
plt.show()
plt.savefig(os.path.join('..', 'Training plots', 'ppg_to_bp_loss.png'),dpi = 300)

In [ ]:
# ─── Load best weights and evaluate on test set ───────────────────────────────
model.load_state_dict(torch.load('best_ppg_to_bp.pth', map_location=device))
model.eval()

preds_norm, trues_norm = [], []
with torch.no_grad():
    for ppg, bp in test_loader:
        ppg = ppg.to(device)
        pred = model(ppg).cpu().numpy()
        preds_norm.append(pred)
        trues_norm.append(bp.numpy())

preds_norm = np.vstack(preds_norm)
trues_norm = np.vstack(trues_norm)

# Inverse-transform to mmHg
preds_mmhg = bp_scaler.inverse_transform(preds_norm)
trues_mmhg = bp_scaler.inverse_transform(trues_norm)

for i, name in enumerate(['SBP', 'DBP']):
    mae  = mean_absolute_error(trues_mmhg[:, i], preds_mmhg[:, i])
    rmse = np.sqrt(mean_squared_error(trues_mmhg[:, i], preds_mmhg[:, i]))
    corr = np.corrcoef(trues_mmhg[:, i], preds_mmhg[:, i])[0, 1]
    print(f'{name} → MAE: {mae:.2f} mmHg | RMSE: {rmse:.2f} mmHg | Pearson r: {corr:.4f}')

In [ ]:
# ─── Scatter plots ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['red', 'blue']
names  = ['SBP', 'DBP']

for i, (ax, name, c) in enumerate(zip(axes, names, colors)):
    ax.scatter(trues_mmhg[:, i], preds_mmhg[:, i], alpha=0.3, s=5, color=c)
    lim = [min(trues_mmhg[:, i].min(), preds_mmhg[:, i].min()),
           max(trues_mmhg[:, i].max(), preds_mmhg[:, i].max())]
    ax.plot(lim, lim, 'k--', label='Ideal')
    ax.set_xlabel(f'True {name} (mmHg)')
    ax.set_ylabel(f'Predicted {name} (mmHg)')
    ax.set_title(f'{name} Prediction')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
plt.savefig(os.path.join('..', 'Training plots', 'ppg_to_bp_scatter.png'),dpi = 300)

print('\nModel saved as: best_ppg_to_bp.pth')
print('BP scaler saved as: bp_scaler.pkl')

## 6. Inference Helper (used by Model 3)

In [ ]:
def predict_bp_from_ppg(ppg_window: np.ndarray, model=model, scaler=bp_scaler) -> dict:
    """
    Given a raw PPG window (625 samples @ 125 Hz), return SBP and DBP in mmHg.
    """
    mu  = ppg_window.mean()
    std = ppg_window.std() + 1e-8
    x   = (ppg_window - mu) / std
    x_t = torch.tensor(x[np.newaxis, np.newaxis, :], dtype=torch.float32).to(device)
    with torch.no_grad():
        pred_norm = model(x_t).cpu().numpy()
    sbp, dbp = scaler.inverse_transform(pred_norm)[0]
    return {'SBP': round(float(sbp), 1), 'DBP': round(float(dbp), 1)}


# ─── Smoke-test using a window from the first cache chunk ─────────────────────
with open(chunk_files[0], 'rb') as f:
    _ppg_s, _sbp_s, _dbp_s = pickle.load(f)

result   = predict_bp_from_ppg(_ppg_s[0])
true_sbp = _sbp_s[0]
true_dbp = _dbp_s[0]
print(f'Sample prediction : SBP={result["SBP"]} mmHg, DBP={result["DBP"]} mmHg')
print(f'True values       : SBP={true_sbp:.1f} mmHg, DBP={true_dbp:.1f} mmHg')
del _ppg_s, _sbp_s, _dbp_s
